In [ ]:

import os
import pandas as pd

print("DATA SPRINT 2.0 - ROAD DAMAGE DETECTION")

print("\nAvailable datasets:")
print(os.listdir("/kaggle/input"))

dataset_path = "/kaggle/input/datasprint-2-road-damage-detection"

if os.path.exists(dataset_path):
    print("\nDataset found successfully!")
    print("\nFiles and folders:")
    print(os.listdir(dataset_path))
else:
    print("\nDataset not found. Please attach the competition dataset.")

# Step1 — Dataset Setup & Verification

In [ ]:

import os
import pandas as pd

dataset_path = "/kaggle/input/competitions/datasprint-2-road-damage-detection"

train_path = os.path.join(dataset_path, "train_images")
val_path = os.path.join(dataset_path, "val_images")
test_path = os.path.join(dataset_path, "test_images")

print("Training Images:", len(os.listdir(train_path)))
print("Validation Images:", len(os.listdir(val_path)))
print("Testing Images:", len(os.listdir(test_path)))

print("\nDataset Loaded Successfully!")

# Step 2 — Training Labels Analysis

In [ ]:

import pandas as pd

train_df = pd.read_csv(dataset_path + "/train_labels.csv")
val_df = pd.read_csv(dataset_path + "/val_labels.csv")

print("Training Labels:", len(train_df))
print("Validation Labels:", len(val_df))

print("\nFirst 5 Rows:")
print(train_df.head())

print("\nDamage Class Distribution:")
print(train_df["class_id"].value_counts().sort_index())

print("\nMissing Values:")
print(train_df.isnull().sum())

# Step 3 — Road Damage Visualization

In [ ]:

import os
import cv2
import matplotlib.pyplot as plt

class_names = {
    0: "Longitudinal Crack",
    1: "Transverse Crack",
    2: "Alligator Crack",
    3: "Pothole"
}
colors = {
    0: (255, 0, 0),
    1: (0, 255, 0),
    2: (0, 0, 255),
    3: (255, 255, 0)
}

sample_ids = train_df["image_id"].drop_duplicates().head(4)

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, image_id in zip(axes.flat, sample_ids):
    image_path = os.path.join(train_path, image_id + ".jpg")
    image = cv2.imread(image_path)

    if image is None:
        ax.set_title("Image not found")
        ax.axis("off")
        continue

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    labels = train_df[train_df["image_id"] == image_id]

    for _, row in labels.iterrows():
        class_id = int(row["class_id"])
        x1 = int(row["x_min"])
        y1 = int(row["y_min"])
        x2 = int(row["x_max"])
        y2 = int(row["y_max"])

        color = colors[class_id]

        cv2.rectangle(image, (x1, y1), (x2, y2), color, 2)

        cv2.putText(
            image,
            class_names[class_id],
            (x1, max(y1 - 10, 15)),
            cv2.FONT_HERSHEY_SIMPLEX,
            0.5,
            color,
            2
        )

    ax.imshow(image)
    ax.set_title(image_id)
    ax.axis("off")

plt.tight_layout()
plt.show()

# Step 4 — GPU Check

In [ ]:

import torch
import importlib.util

print("PyTorch Version:", torch.__version__)
print("GPU Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("GPU not available")

if importlib.util.find_spec("ultralytics"):
    print("Ultralytics installed")
else:
    print("Ultralytics not installed")

# Step 5 — Install YOLOv8

In [ ]:
%pip install -q ultralytics

In [ ]:
from ultralytics import YOLO
import ultralytics

print("Ultralytics Version:", ultralytics.__version__)
print("YOLOv8 Ready!")

# Step 6 — Dataset Conversion

In [ ]:
import os
import cv2
import pandas as pd
from tqdm.auto import tqdm

output_path = "/kaggle/working/road_damage"

def convert_labels(csv_file, image_folder, split):
    df = pd.read_csv(csv_file)

    label_folder = os.path.join(output_path, "labels", split)
    os.makedirs(label_folder, exist_ok=True)

    grouped = df.groupby("image_id")

    image_files = sorted(
        f for f in os.listdir(image_folder)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    )

    for filename in tqdm(image_files, desc=split):
        image_id = os.path.splitext(filename)[0]
        image = cv2.imread(os.path.join(image_folder, filename))

        if image is None:
            raise ValueError("Cannot read image: " + filename)

        height, width = image.shape[:2]
        lines = []

        if image_id in grouped.groups:
            rows = grouped.get_group(image_id)

            for _, row in rows.iterrows():
                x1 = float(row["x_min"])
                y1 = float(row["y_min"])
                x2 = float(row["x_max"])
                y2 = float(row["y_max"])

                x_center = ((x1 + x2) / 2) / width
                y_center = ((y1 + y2) / 2) / height
                box_width = (x2 - x1) / width
                box_height = (y2 - y1) / height

                class_id = int(row["class_id"])

                lines.append(
                    f"{class_id} {x_center:.6f} {y_center:.6f} "
                    f"{box_width:.6f} {box_height:.6f}"
                )

        label_file = os.path.join(label_folder, image_id + ".txt")

        with open(label_file, "w") as f:
            f.write("\n".join(lines))

convert_labels(
    dataset_path + "/train_labels.csv",
    train_path,
    "train"
)

convert_labels(
    dataset_path + "/val_labels.csv",
    val_path,
    "val"
)

print("YOLO label conversion completed!")
print("Training label files:", len(os.listdir(output_path + "/labels/train")))
print("Validation label files:", len(os.listdir(output_path + "/labels/val")))

# Step 7 — YOLO Dataset Configuration

In [ ]:
import os

data_yaml = "/kaggle/working/road_damage/data.yaml"

yaml_content = f"""
train: {train_path}
val: {val_path}

train_labels: {output_path}/labels/train
val_labels: {output_path}/labels/val

nc: 4

names:
  0: D00
  1: D10
  2: D20
  3: D40
"""

with open(data_yaml, "w") as f:
    f.write(yaml_content)

print("data.yaml created successfully!")
print(open(data_yaml).read())

# Step 8 — Dataset Setup and YAML Configuration

In [ ]:
import os

base = "/kaggle/working/road_damage"

images_dir = os.path.join(base, "images")
os.makedirs(images_dir, exist_ok=True)

for split, source in [("train", train_path), ("val", val_path)]:
    link = os.path.join(images_dir, split)

    if not os.path.lexists(link):
        os.symlink(source, link)

yaml_content = f"""
path: {base}
train: images/train
val: images/val

nc: 4
names:
  0: D00
  1: D10
  2: D20
  3: D40
"""

with open(os.path.join(base, "data.yaml"), "w") as f:
    f.write(yaml_content)

print("Dataset setup completed!")

for split in ["train", "val"]:
    image_folder = os.path.join(base, "images", split)
    label_folder = os.path.join(base, "labels", split)

    print(split, "images:", len(os.listdir(image_folder)))
    print(split, "labels:", len(os.listdir(label_folder)))

print("\nConfiguration:")
print(open(os.path.join(base, "data.yaml")).read())

# Step 9 — Training Code

In [ ]:

import os
import torch
from pathlib import Path
from ultralytics import YOLO, settings

settings.update({"raytune": False})

source = Path("/kaggle/input/competitions/datasprint-2-road-damage-detection")
base = Path("/kaggle/working/road_damage")

for split in ["train", "val"]:
    image_source = source / f"{split}_images"
    image_target = base / "images" / split
    label_target = base / "labels" / split

    if image_target.is_symlink():
        image_target.unlink()

    image_target.mkdir(parents=True, exist_ok=True)

    for image_file in image_source.glob("*.jpg"):
        link = image_target / image_file.name

        if not link.exists():
            link.symlink_to(image_file)

    print(split, "images:", len(list(image_target.glob("*.jpg"))))
    print(split, "labels:", len(list(label_target.glob("*.txt"))))

yaml_file = base / "data.yaml"

yaml_file.write_text(
    f"path: {base}\n"
    "train: images/train\n"
    "val: images/val\n"
    "nc: 4\n"
    "names: [D00, D10, D20, D40]\n"
)

sample_image = base / "images/train/img_000005.jpg"
sample_label = base / "labels/train/img_000005.txt"

print("\nSample image exists:", sample_image.exists())
print("Sample label exists:", sample_label.exists())
print("Sample label:", sample_label.read_text() if sample_label.exists() else "Missing")

assert sample_image.exists()
assert sample_label.exists()
assert sample_label.read_text().strip()

print("\nGPU:", torch.cuda.get_device_name(0))
print("Dataset ready for training")

model = YOLO("yolov8n.pt")

results = model.train(
    data=str(yaml_file),
    epochs=5,
    imgsz=640,
    batch=16,
    device=0,
    workers=2,
    project=str(base / "runs"),
    name="yolov8_baseline",
    exist_ok=True,
    plots=True
)

best_model = base / "runs/yolov8_baseline/weights/best.pt"

print("\nTraining completed!")
print("Best model:", best_model)
print("Model saved:", best_model.exists())

In [ ]:
import os

dataset_path = "/kaggle/input/competitions/datasprint-2-road-damage-detection"

files = [
    "make_kaggle_submission.py",
    "check_submission.py"
]

for filename in files:
    file_path = os.path.join(dataset_path, filename)

    print("\nFILE:", filename)
    print("=" * 50)

    with open(file_path, "r") as f:
        print(f.read())

# Step 11 — Test Prediction

In [ ]:
import os
import sys
import csv
import subprocess
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from ultralytics import YOLO

# Paths
dataset = Path("/kaggle/input/competitions/datasprint-2-road-damage-detection")
test_dir = dataset / "test_images"
ids_file = dataset / "test_image_ids.csv"

model_path = Path(
    "/kaggle/working/road_damage/runs/yolov8_baseline/weights/best.pt"
)

pred_file = Path("/kaggle/working/test_predictions.csv")
submission_file = Path("/kaggle/working/submission.csv")

assert model_path.exists(), "Trained model not found!"
assert ids_file.exists(), "Test image IDs file not found!"

# Load model
model = YOLO(str(model_path))

# Read test IDs
test_ids = pd.read_csv(ids_file)["image_id"].astype(str).tolist()

print("Total test images:", len(test_ids))

# Generate detailed predictions
total_boxes = 0
missing_images = []

with open(pred_file, "w", newline="") as f:
    writer = csv.writer(f)

    writer.writerow([
        "image_id", "class_id", "score",
        "x_min", "y_min", "x_max", "y_max"
    ])

    for image_id in tqdm(test_ids, desc="Predicting"):
        image_path = test_dir / f"{image_id}.jpg"

        if not image_path.exists():
            missing_images.append(image_id)
            continue

        results = model.predict(
            source=str(image_path),
            imgsz=640,
            conf=0.001,
            iou=0.7,
            max_det=100,
            device=0,
            verbose=False
        )

        boxes = results[0].boxes

        if boxes is None:
            continue

        for box in boxes:
            class_id = int(box.cls.item())
            score = float(box.conf.item())

            x1, y1, x2, y2 = box.xyxy[0].tolist()

            writer.writerow([
                image_id,
                class_id,
                round(score, 5),
                round(x1, 1),
                round(y1, 1),
                round(x2, 1),
                round(y2, 1)
            ])

            total_boxes += 1

print("Total detections:", total_boxes)
print("Missing images:", len(missing_images))

# Use official competition submission converter
convert_command = [
    sys.executable,
    str(dataset / "make_kaggle_submission.py"),
    "--pred", str(pred_file),
    "--image-ids", str(ids_file),
    "--out", str(submission_file)
]

subprocess.run(convert_command, check=True)

# Validate using official competition checker
check_command = [
    sys.executable,
    str(dataset / "check_submission.py"),
    "--pred", str(submission_file),
    "--image-ids", str(ids_file)
]

subprocess.run(check_command, check=True)

# Preview
submission = pd.read_csv(submission_file)

print("\nSubmission Preview:")
print(submission.head())

print("\nTotal rows:", len(submission))
print("Submission saved:", submission_file)

In [ ]:
from ultralytics import YOLO
from IPython.display import display
from PIL import Image

model = YOLO(
    "/kaggle/working/road_damage/runs/yolov8_baseline/weights/best.pt"
)

image_path = "/kaggle/input/competitions/datasprint-2-road-damage-detection/test_images/img_000001.jpg"

results = model.predict(
    source=image_path,
    conf=0.25,
    imgsz=640,
    device=0
)

display(Image.fromarray(results[0].plot()[..., ::-1]))

In [ ]:

import os
import random
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from ultralytics import YOLO

# 1. Load trained model
model_path = "/kaggle/working/road_damage/runs/yolov8_baseline/weights/best.pt"

if not os.path.exists(model_path):
    raise FileNotFoundError("best.pt not found. Check training output.")

model = YOLO(model_path)

# 2. Damage class names
class_names = {
    0: "Longitudinal Crack",
    1: "Transverse Crack",
    2: "Alligator Crack",
    3: "Pothole"
}

# 3. Test images folder
test_path = Path(
    "/kaggle/input/competitions/datasprint-2-road-damage-detection/test_images"
)

images = sorted(test_path.glob("*.jpg"))

if not images:
    raise FileNotFoundError("No test images found.")

# 4. Select 5 random images
random.seed(42)
selected_images = random.sample(images, min(5, len(images)))

# 5. Predict and visualize
fig, axes = plt.subplots(
    1, len(selected_images),
    figsize=(25, 6)
)

if len(selected_images) == 1:
    axes = [axes]

colors = {
    0: (255, 80, 80),
    1: (80, 200, 255),
    2: (255, 180, 50),
    3: (50, 220, 100)
}

for ax, image_path in zip(axes, selected_images):

    result = model.predict(
        source=str(image_path),
        conf=0.25,
        imgsz=640,
        device=0,
        verbose=False
    )[0]

    image = cv2.imread(str(image_path))
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    detections = 0

    for box in result.boxes:

        cls = int(box.cls.item())
        confidence = float(box.conf.item())

        x1, y1, x2, y2 = map(
            int, box.xyxy[0].tolist()
        )

        color = colors[cls]

        # Bounding box
        cv2.rectangle(
            image,
            (x1, y1),
            (x2, y2),
            color,
            3
        )

        label = f"{class_names[cls]} {confidence:.1%}"

        # Label background
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.5
        thickness = 1

        (tw, th), baseline = cv2.getTextSize(
            label, font, font_scale, thickness
        )

        label_y = max(y1, th + 10)

        cv2.rectangle(
            image,
            (x1, label_y - th - 8),
            (x1 + tw + 6, label_y + baseline),
            color,
            -1
        )

        cv2.putText(
            image,
            label,
            (x1 + 3, label_y - 4),
            font,
            font_scale,
            (0, 0, 0),
            thickness,
            cv2.LINE_AA
        )

        detections += 1

    ax.imshow(image)
    ax.set_title(
        f"{image_path.name}\nDetections: {detections}",
        fontsize=11
    )
    ax.axis("off")

plt.tight_layout()
plt.show()

print("Prediction!")
